In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/09/21 15:14:23 WARN Utils: Your hostname, GANIU-ODEYINKA resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/21 15:14:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/21 15:14:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/21 15:14:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [ ]:
'''
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS no_of_records
    
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
'''

In [3]:
df_green.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-17 22:01:22|  2020-01-17 22:08:44|                 N|         1|          97|         181|              1|         1.12|        6.5|  0.5|    0.

In [8]:
df_green.rdd.take(10)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 17, 22, 1, 22), lpep_dropoff_datetime=datetime.datetime(2020, 1, 17, 22, 8, 44), store_and_fwd_flag='N', RatecodeID=1, PULocationID=97, DOLocationID=181, passenger_count=1, trip_distance=1.12, fare_amount=6.5, extra=0.5, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=7.8, payment_type=2, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 25, 23, 13, 37), lpep_dropoff_datetime=datetime.datetime(2020, 1, 25, 23, 21, 53), store_and_fwd_flag='N', RatecodeID=1, PULocationID=65, DOLocationID=49, passenger_count=1, trip_distance=1.5, fare_amount=7.5, extra=0.5, mta_tax=0.5, tip_amount=1.76, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=10.56, payment_type=1, trip_type=1, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 12, 10, 48, 36), lpep_dropoff_da

In [4]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [5]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 17, 22, 1, 22), PULocationID=97, total_amount=7.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 25, 23, 13, 37), PULocationID=65, total_amount=10.56),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 12, 10, 48, 36), PULocationID=244, total_amount=9.96),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 17, 51, 58), PULocationID=75, total_amount=11.05),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 6, 22, 53, 17), PULocationID=260, total_amount=20.3)]

In [20]:
#to perform where/filter by datetime

from datetime import datetime

start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start


In [21]:
rows = rdd.take(10)
row = rows[0]
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 17, 22, 1, 22), PULocationID=97, total_amount=7.8)

In [22]:
#to perform groupby using key/value

def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [23]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [24]:
from collections import namedtuple

In [25]:
revenue_row = namedtuple('revenue_row', ['hour', 'zone', 'revenue', 'count'])

In [27]:
def unwrap(row):
    return revenue_row(
        hour = row[0][0], 
        zone = row[0][1], 
        revenue = row[1][0],
        count = row[1][1]
    )

In [28]:
from pyspark.sql import types

In [34]:
df_result.schema

StructType([StructField('hour', TimestampType(), True), StructField('zone', IntegerType(), True), StructField('revenue', DoubleType(), True), StructField('count', IntegerType(), True)])

In [29]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [30]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

#reduceByKey takes (key, value) as input and return (key, reduced_value)

In [32]:
df_result.show()

[Stage 11:==================================================>       (7 + 1) / 8]

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-25 23:00:00|  65|            366.23|   19|
|2020-01-03 00:00:00|  82| 335.6500000000001|   27|
|2020-01-25 09:00:00| 259|             90.22|    2|
|2020-01-31 14:00:00|  95|496.75000000000017|   34|
|2020-01-06 09:00:00|  95|            590.49|   33|
|2020-01-14 12:00:00|  97|            176.08|   12|
|2020-01-18 21:00:00| 130|             84.16|    7|
|2020-01-10 14:00:00| 128|              32.3|    2|
|2020-01-04 17:00:00| 129| 524.8000000000001|   28|
|2020-01-22 10:00:00|  33| 591.6700000000001|   27|
|2020-01-22 10:00:00|  97|             232.4|   10|
|2020-01-28 21:00:00|  77|            138.86|    5|
|2020-01-28 14:00:00| 185|             43.28|    2|
|2020-01-07 08:00:00| 145|172.01000000000002|   16|
|2020-01-29 19:00:00|  95| 731.4199999999998|   52|
|2020-01-02 13:00:00| 131|145.04999999999998|    4|
|2020-01-23 

In [35]:
df_result.write.parquet('tmp/green-revenue')

25/09/21 15:32:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [ ]:
#mapPartitions

In [36]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [37]:
import pandas as pd

In [38]:
rows = duration_rdd.take(10)

In [39]:
df = pd.DataFrame(rows, columns=columns)

In [40]:
columns

['VendorID',
 'lpep_pickup_datetime',
 'PULocationID',
 'DOLocationID',
 'trip_distance']

In [41]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [42]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row

In [43]:
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch)\
    .toDF() \
    .drop('Index')

In [44]:
df_predicts \
    .select('predicted_duration') \
    .show()

[Stage 17:>                                                         (0 + 1) / 1]

+-------------------+
| predicted_duration|
+-------------------+
| 5.6000000000000005|
|                7.5|
|                7.4|
|               4.55|
|              21.25|
|               24.8|
| 62.199999999999996|
|                2.0|
| 5.3500000000000005|
|                8.0|
| 27.450000000000003|
| 55.199999999999996|
|               39.5|
| 13.600000000000001|
| 14.450000000000001|
|               7.95|
|               8.95|
|-10.549999999999999|
|               51.5|
| 42.599999999999994|
+-------------------+
only showing top 20 rows

